In [ ]:
import sqlite3
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("../data/raw")

df_p = pd.read_csv(DATA_DIR / "patient.csv")
df_t = pd.read_csv(DATA_DIR / "treatment.csv")
df_v = pd.read_csv(DATA_DIR / "vitalPeriodic.csv")
df_h = pd.read_csv(DATA_DIR / "hospital.csv")
df_d = pd.read_csv(DATA_DIR / "diagnosis.csv")

In [ ]:
mask = df_t["treatmentstring"].str.contains("ventilation", case=False, na=False)
total = mask.sum()
print(f"Trattamenti con ventilazione: {total}")

#unici per pazienti
total_pazienti = df_t[mask]["patientunitstayid"].nunique()
print(f"Pazienti con ventilazione: {total_pazienti}")



Trattamenti con ventilazione: 3718
Pazienti con ventilazione: 826


In [ ]:
# Crea colonna ventilato
ventilati = df_t[df_t["treatmentstring"].str.contains(
    "mechanical ventilation", case=False, na=False
)]["patientunitstayid"].unique()

df_p["ventilato"] = df_p["patientunitstayid"].isin(ventilati).astype(int)

# Distribuzione morti per gruppo
dist = df_p.groupby("ventilato")["unitdischargestatus"].value_counts()
print(dist)

# Mortalità % per gruppo
print("\nMortalità per gruppo:")
print(df_p.groupby("ventilato")["unitdischargestatus"]
      .apply(lambda x: (x == "Expired").mean() * 100)
      .rename({0: "Non ventilati", 1: "Ventilati"}))

ventilato  unitdischargestatus
0          Alive                  2017
           Expired                  65
1          Alive                   375
           Expired                  61
Name: count, dtype: int64

Mortalità per gruppo:
ventilato
Non ventilati     3.119002
Ventilati        13.990826
Name: unitdischargestatus, dtype: float64


In [ ]:
# Definizione trattamento A entro 24h (1440 minuti)
FINESTRA = 1440  # minuti

# Filtra VM invasiva nella finestra
vm_mask = (
    df_t["treatmentstring"].str.contains("mechanical ventilation", case=False, na=False) &
    (df_t["treatmentoffset"] <= FINESTRA)
)

pazienti_ventilati = df_t[vm_mask]["patientunitstayid"].unique()

# Crea colonna trattamento nel dataset paziente
df_p["A"] = df_p["patientunitstayid"].isin(pazienti_ventilati).astype(int)

# Outcome Y
df_p["Y"] = (df_p["unitdischargestatus"] == "Expired").astype(int)

# Verifica
print(df_p[["A", "Y"]].value_counts())
print(f"\nTreatati (A=1): {df_p['A'].sum()}")
print(f"ATE grezzo (differenza semplice): {df_p[df_p['A']==1]['Y'].mean() - df_p[df_p['A']==0]['Y'].mean():.3f}")

A  Y
0  0    2050
1  0     344
0  1      72
1  1      54
Name: count, dtype: int64

Treatati (A=1): 398
ATE grezzo (differenza semplice): 0.102
